## Modules import

In [1]:
import pandas as pd
import json
import time
import clickhouse_connect
import plotly.express as px

## Connection to ClickHouse

### Dataset info

Data being analyzed here is a part of old commercial advertisements logs, provided to me as a processable example for the task. It is not covered by NDA and has neglectable business value.

Dataset includes 2 tables: `E.TrackDays` (13170 rows) and `E.Tracking3` (296979 rows) - of structure and settings described below.

#### Table 1 - E.TrackDays

##### Description

**Purpose:** Aggregated daily advertising statistics.

**Engine:** `SummingMergeTree`. 

##### Columns

**Dimensions:**

| Field           | Type     | Description |
|-----------------|----------|-------------|
| `dt`            | `Date`   | Event date |
| `demand_id`     | `UInt32` | Demand source ID |
| `demand_name`   | `String` | Demand source name |
| `router_id`     | `UInt32` | Request router ID |
| `router_uid`    | `String` | Router unique string ID |
| `router_name`   | `String` | Router name |
| `tag_id`        | `UInt32` | Placement (tag) ID |
| `tag_uid`       | `String` | Tag unique string ID |
| `tag_name`      | `String` | Tag name |
| `channel_id`    | `UInt32` | Sales channel / traffic source ID |
| `channel_uid`   | `String` | Channel unique string ID |
| `channel_name`  | `String` | Channel name |
| `campaign_id`   | `UInt32` | Campaign ID |
| `campaign_uid`  | `String` | Campaign unique string ID |
| `campaign_name` | `String` | Campaign name |
| `creative_id`   | `UInt32` | Creative ID |
| `creative_uid`  | `String` | Creative unique string ID |
| `creative_name` | `String` | Creative name |
| `adv_id`        | `UInt32` | Advertiser ID |
| `adv_name`      | `String` | Advertiser name |
| `sub_id`        | `UInt32` | Sub-advertiser / agency ID |
| `sub_name`      | `String` | Sub-advertiser name |
| `app_name`      | `String` | Application / site name |
| `app_bundle`    | `String` | Application bundle ID |
| `country`       | `String` | Country (geo) |
| `device`        | `String` | Device type |
| `os`            | `String` | Operating system |
| `size`          | `String` | Placement size |
| `reason`        | `String` | Reason / event type code |
| `device2`       | `String` | Additional device detail |
| `req_type`      | `Enum8(''=0,'VAST'=1,'RTB'=2)` | Request protocol |

**Prices:**

| Field | Type | Description |
|-------|------|-------------|
| `system_price` | `Decimal(18,10)` | System price (e.g. base rate) |
| `tag_price` | `Decimal(18,8)` | Price for the placement |
| `channel_price` | `Decimal(18,8)` | Price for the channel |

**Metrics:**

All metrics are `UInt64` and are summed automatically.

| Field | Description |
|-------|-------------|
| `reqs` | Ad requests |
| `req2` | Additional request counter (e.g. filtered) |
| `opps` | Opportunities to serve |
| `opp2` | Refined opportunities |
| `ress` | Responses |
| `res2` | Additional response counter |
| `imps` | Impressions |
| `invs` | In‑view / engagements |
| `opp3` | Extra opportunities |
| `opp3t` | Total extra opportunities |

**Key Settings**  
- `PARTITION BY toYYYYMM(dt)`  
- `PRIMARY KEY (demand_id, dt)`  
- `ORDER BY (demand_id, dt, demand_name, tag_id, tag_uid, tag_name, channel_id, channel_uid, channel_name, adv_id, adv_name, sub_id, sub_name, app_name, app_bundle, country, device, os, size, reason, device2, router_id, campaign_id, creative_id)`  
- `index_granularity = 8192` 

> **Important:** Always query with `GROUP BY` on the full sorting key and `SUM()` on metrics to obtain accurate results (parts may not be fully merged).

#### Table 2 - E.Tracking3

##### Description

**Purpose:** Aggregated real‑time advertising tracking statistics.  

**Engine:** `SummingMergeTree`

##### Columns

**Dimensions:**

| Field | Type | Description |
|-------|------|-------------|
| `dt` | `DateTime` | Event timestamp |
| `demand_id` | `UInt32` | Demand source ID |
| `demand_name` | `String` | Demand source name |
| `router_id` | `UInt32` | Request router ID |
| `router_uid` | `String` | Router unique string ID |
| `router_name` | `String` | Router name |
| `tag_id` | `UInt32` | Placement (tag) ID |
| `tag_uid` | `String` | Tag unique string ID |
| `tag_name` | `String` | Tag name |
| `adv_id` | `UInt32` | Advertiser ID |
| `adv_name` | `String` | Advertiser name |
| `channel_id` | `UInt32` | Sales channel / traffic source ID |
| `channel_uid` | `String` | Channel unique string ID |
| `channel_name` | `String` | Channel name |
| `campaign_id` | `UInt32` | Campaign ID |
| `campaign_uid` | `String` | Campaign unique string ID |
| `campaign_name` | `String` | Campaign name |
| `creative_id` | `UInt32` | Creative ID |
| `creative_uid` | `String` | Creative unique string ID |
| `creative_name` | `String` | Creative name |
| `sub_id` | `UInt32` | Sub-advertiser / agency ID |
| `sub_name` | `String` | Sub-advertiser name |
| `app_name` | `String` | Application name |
| `app_bundle` | `String` | Application bundle ID |
| `app_url` | `String` | Application URL |
| `site_name` | `String` | Website name |
| `site_domain` | `String` | Website domain |
| `site_url` | `String` | Website URL |
| `dooh_id` | `String` | Digital out‑of‑home identifier |
| `venue_type_id` | `UInt32` | Venue type ID (DOOH) |
| `country` | `String` | Country (geo) |
| `state` | `String` | State / region |
| `city` | `String` | City |
| `zip` | `String` | ZIP / postal code |
| `device` | `String` | Device type |
| `os` | `String` | Operating system |
| `size` | `String` | Placement size (e.g. 320x50) |
| `reqSize` | `String` | Requested size |
| `reason` | `String` | Reason / status code |
| `device2` | `String` | Additional device detail |
| `pubid` | `String` | Publisher ID |
| `pubid2` | `String` | Additional publisher ID |
| `iab` | `String` | IAB category |
| `crid` | `String` | Creative ID (external) |
| `contentCategory` | `String` | Content category |
| `contentTitle` | `String` | Content title |
| `contentNetwork` | `String` | Content network |
| `contentGenre` | `String` | Content genre |
| `contentLanguage` | `String` | Content language |
| `contentRating` | `String` | Content rating |
| `contentChannel` | `String` | Content channel |
| `coppa` | `UInt8` | COPPA flag (0 or 1) |
| `req_type` | `Enum8(''=0,'VAST'=1,'RTB'=2)` | Request protocol |
| `imp_type` | `String` | Impression type |

**Prices & Bids:**

| Field | Type | Description |
|-------|------|-------------|
| `system_price` | `Decimal(18,10)` | System price (base rate) |
| `tag_price` | `Decimal(18,8)` | Price for the placement |
| `tag_cpm` | `Decimal(9,4)` | Tag CPM |
| `channel_price` | `Decimal(18,8)` | Price for the channel |
| `channel_cpm` | `Decimal(9,4)` | Channel CPM |
| `bidFloor` | `Decimal(18,8)` | Bid floor price |
| `multiplier` | `Decimal(18,2)` | Multiplier applied |

**Metrics:**

All metrics are `UInt64` and are summed automatically.

| Field | Description |
|-------|-------------|
| `reqs` | Ad requests |
| `opps` | Opportunities to serve |
| `ress` | Responses |
| `imps` | Impressions |
| `req2` | Additional request counter (e.g. filtered) |
| `opp2` | Refined opportunities |
| `res2` | Additional response counter |
| `opp3` | Extra opportunities |
| `opp3s` | Extra opportunities (supply) |
| `opp3m` | Extra opportunities (mediation) |
| `opp3t` | Total extra opportunities |
| `req3` | Additional request counter |
| `wins` | Won bids |
| `invs` | In‑view / engagements |
| `event_0` | Event: start |
| `event_25` | Event: 25% progress |
| `event_50` | Event: 50% progress |
| `event_75` | Event: 75% progress |
| `event_100` | Event: 100% completion |
| `clicks` | Clicks |

**Key Settings:**  
- `PARTITION BY toYYYYMMDD(dt)` - daily partitions  
- `PRIMARY KEY demand_id`  
- `ORDER BY (demand_id, demand_name, tag_id, tag_uid, tag_name, tag_cpm, channel_id, channel_uid, channel_name, channel_cpm, adv_id, adv_name, sub_id, sub_name, app_name, app_bundle, country, device, os, size, reason, device2, dt, iab, pubid, crid, contentCategory, contentTitle, contentNetwork, contentGenre, coppa, pubid2, reqSize, imp_type, site_name, site_domain, state, city, zip, router_id, campaign_id, creative_id, dooh_id, venue_type_id, contentLanguage, contentRating, contentChannel)`  
- `storage_policy = 'tracking3'`  
- `index_granularity = 8192`  

> **Important:** Always query with `GROUP BY` on the full sorting key and `SUM()` on metrics to obtain accurate results (parts may not be fully merged).

### Connection to database

In [2]:
with open("logging_config.json", "r") as logging_config_file:
    logging_config = json.load(logging_config_file)

In [3]:
click_client = clickhouse_connect.get_client(
    host=logging_config["host"],
    port=logging_config["port"],
    username=logging_config["user"],
    password=logging_config["password"],
    database=logging_config["database"],
    connect_timeout=60,
    send_receive_timeout=600
)

## Sessions per user count

### Definitions

For the role of `users` we consider `channels`, distinguished by unique `channel_id`.

For the role of `sessions` we consider periods of time during which `requests` were above `1000` constantly.

In [4]:
tracking3_name = "Tracking3"
trackdays_name = "TrackDays"

### SPU count

In [ ]:
def get_sessions_per_user(client, 
                          table: str,
                          user_ids_column: str,
                          session_condition: str,
                          date_from: str | None = None,
                          date_to: str | None = None) -> pd.DataFrame:
    """
    Calculates average number of sessions for every user
    
    Parameters:
        client:            ClickHouse client created with `clickhouse_connect` module
        table:             table name
        user_ids_column:   column with user unique (1 id to 1 user) identificators
        session_condition: condition to define session
        date_from:         date from which should sessions be calculated
        date_to:           date up to which should sessions be calculated
        
    Returns:
        DataFrame with columns `[ user_ids_column, "n_sessions"]`
    """    
    date_filter = ""
    if date_from is not None:
        date_filter += f" AND dt >= parseDateTimeBestEffort('{date_from}')"
    if date_to is not None:
        date_filter += f" AND dt < parseDateTimeBestEffort('{date_to}') + INTERVAL 1 DAY"
    
    query_spu = f"""
        SELECT users.user_id, coalesce(sessions.session_count, 0) AS sessions_per_user
            FROM (
                SELECT DISTINCT {user_ids_column} AS user_id
                    FROM {table}
                    WHERE 1=1 {date_filter}
            ) users
            LEFT JOIN (
                SELECT user_id, countDistinct(session_group) AS session_count
                    FROM (
                        SELECT {user_ids_column} AS user_id,
                            row_number() OVER (
                                PARTITION BY {user_ids_column} 
                                ORDER BY dt
                            ) - row_number() OVER (
                                PARTITION BY {user_ids_column}, ({session_condition}) 
                                ORDER BY dt
                            ) AS session_group,
                            reqs
                        FROM {table}
                        WHERE 1=1 {date_filter}
                    ) t_alias
                    WHERE {session_condition}
                    GROUP BY user_id
            ) sessions 
            ON users.user_id = sessions.user_id
            ORDER BY users.user_id
    """
    return pd.DataFrame(
        client.query(query_spu).result_rows, 
        columns=[user_ids_column, "n_sessions"]
    )


In [6]:
spu_tracking3 = get_sessions_per_user(
    click_client,
    tracking3_name,
    "channel_id",
    "reqs >= 1000"
)
spu_tracking3

,channel_id,n_sessions
0,0,0
1,115,208
2,116,271
3,117,267
4,119,141
5,120,225
6,121,109
7,122,405
8,129,105
9,132,226


In [11]:
spu_tracking3 = spu_tracking3[ spu_tracking3["channel_id"] > 0 ]

### SPU visualization

In [12]:
fig = px.bar(
    spu_tracking3, 
    x="channel_id",
    y="n_sessions",
    title=f"Sessions per channel distribution for table {tracking3_name}",
    labels={
        "channel_id": "Channel id", 
        "n_sessions": "Session number"
    }
)
fig.show()

## Aggregation time with & without sorting key

In [8]:
def measure_query_time(client, 
                       query: str,
                       verbose: bool | None = True):
    start = time.time()
    client.query(query)
    query_time = time.time() - start

    if verbose:
        print(f"Query took time: {query_time:.5f} ms")
    return query_time

In [9]:
time_condition = "dt >= '2025-01-01' AND dt <= '2025-01-31'"

query_sort = f"""
    SELECT demand_id, dt, sum(imps) AS total_imps
        FROM {trackdays_name}
        WHERE {time_condition}
        GROUP BY demand_id, dt
        ORDER BY demand_id, dt
"""
query_no_sort = f"""
    SELECT app_name, sum(imps) AS total_imps
        FROM {trackdays_name}
        WHERE {time_condition}
        GROUP BY app_name
        ORDER BY app_name
"""

In [10]:
print("Aggregation on sorting key example:")
time_sort = measure_query_time(click_client, query_sort)

print("\nAggregation on non-key row example:")
time_no_sort = measure_query_time(click_client, query_no_sort)

print(f"\nSorting key usage speedup:\n"
      f"    times:      x{time_sort / time_no_sort:.2f},\n"
      f"    difference: {time_sort - time_no_sort:.2f} ms")

Aggregation on sorting key example:
Query took time: 0.26083 ms

Aggregation on non-key row example:
Query took time: 0.06171 ms

Sorting key usage speedup:
    times:      x4.23,
    difference: 0.20 ms
